"# Phase 5 — Migration Zone Discovery\n"

## Step 0 — Install & Import Libraries


In [1]:
%pip install plotly pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, silhouette_samples, davies_bouldin_score
)
from math import radians, sin, cos, sqrt, atan2
import warnings
warnings.filterwarnings('ignore')

COLORS  = {'Eric': '#065A82', 'Nico': '#02C39A', 'Sanne': '#F4A261'}
BIRDS   = ['Eric', 'Nico', 'Sanne']
SEED    = 42

EXPECTED_ZONES = [
    {'name': 'Netherlands',     'lat_band': (50, 53)},
    {'name': 'Central Europe',  'lat_band': (42, 50)},
    {'name': 'Iberia/Spain',    'lat_band': (36, 42)},
    {'name': 'North Africa',    'lat_band': (28, 36)},
    {'name': 'Sahara transit',  'lat_band': (18, 28)},
    {'name': 'West Africa',     'lat_band': (4, 18)},
]


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Load Dataset


In [2]:
df = pd.read_csv('bird_migration_features.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

df['month'] = df['date_time'].dt.month
df['hour']  = df['date_time'].dt.hour

def encode_season(row):
    m, y = row['month'], row['date_time'].year
    if y == 2013 and m in [8,9,10,11]: return 1
    if (y == 2013 and m == 12) or (y == 2014 and m == 1): return 2
    return 3
df['season'] = df.apply(encode_season, axis=1)

df['is_resting'] = (df['speed_2d'] < 1.0).astype(int)

df['altitude_clipped'] = df['altitude'].clip(-100, 1000)

print('=== DATASET LOADED ===')
print(f'Total records  : {len(df):,}')
print(f'Birds          : {df["bird_name"].unique().tolist()}')
print(f'Date range     : {df["date_time"].min().date()} → {df["date_time"].max().date()}')
print(f'Null values    : {df[["latitude","longitude"]].isnull().sum().sum()} in lat/lon')
print(f'Resting records: {df["is_resting"].sum():,} ({df["is_resting"].mean()*100:.1f}%)')
print(f'\nRecords per bird:')
print(df['bird_name'].value_counts())

=== DATASET LOADED ===
Total records  : 61,920
Birds          : ['Eric', 'Nico', 'Sanne']
Date range     : 2013-08-15 → 2014-04-30
Null values    : 0 in lat/lon
Resting records: 28,021 (45.3%)

Records per bird:
bird_name
Nico     21121
Sanne    21004
Eric     19795
Name: count, dtype: int64


## Step 2 — Prepare Geographic Input (lat/lon → Radians)


In [3]:
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()
print(f'Records after null drop: {len(df_clean):,} (dropped: {len(df)-len(df_clean)})')

coords = df_clean[['latitude', 'longitude']].values
coords_rad = np.radians(coords)


Records after null drop: 61,920 (dropped: 0)


## Step 3 — Resting-Point Deduplication


In [4]:
df_clean['lat_round'] = df_clean['latitude'].round(2)
df_clean['lon_round'] = df_clean['longitude'].round(2)

def mark_resting_duplicates(bird_df):
    bird_df = bird_df.copy()
    lat_shifted = bird_df['lat_round'].shift()
    lon_shifted = bird_df['lon_round'].shift()
    same_pos    = (bird_df['lat_round'] == lat_shifted) & \
                  (bird_df['lon_round'] == lon_shifted)
    resting     = bird_df['is_resting'] == 1
    bird_df['is_rest_dup'] = (same_pos & resting).astype(int)
    return bird_df

bird_name_series = df_clean['bird_name'].copy()
df_clean = df_clean.groupby('bird_name', group_keys=False).apply(mark_resting_duplicates)
df_clean['bird_name'] = bird_name_series

df_for_fit  = df_clean[df_clean['is_rest_dup'] == 0].copy()
coords_fit  = df_for_fit[['latitude', 'longitude']].values
coords_fit_rad = np.radians(coords_fit)
scaler = StandardScaler()
coords_fit_rad = scaler.fit_transform(coords_fit_rad)

print(f'Original records     : {len(df_clean):,}')
print(f'Resting duplicates   : {df_clean["is_rest_dup"].sum():,}')
print(f'Records used for fit : {len(df_for_fit):,}')
print(f'Reduction            : {(1 - len(df_for_fit)/len(df_clean))*100:.1f}%')
print(f'\nNote: cluster_id will be assigned back to ALL {len(df_clean):,} records')
print('using the fitted model — deduplication only affects centroid calculation.')

Original records     : 61,920
Resting duplicates   : 21,932
Records used for fit : 39,988
Reduction            : 35.4%

Note: cluster_id will be assigned back to ALL 61,920 records
using the fitted model — deduplication only affects centroid calculation.


## Step 4 — Elbow Method (k = 2 to 12)


In [5]:
K_RANGE    = range(2, 13)
inertias   = []
sil_scores = []
db_scores  = []
km_models  = {}

for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=10,
        max_iter=300,
        init='k-means++'
    )
    labels_fit = km.fit_predict(coords_fit_rad)
    inertias.append(km.inertia_)
    km_models[k] = km

    n_sample = min(5000, len(coords_fit_rad))
    idx_s    = np.random.RandomState(SEED).choice(len(coords_fit_rad),
                                                   size=n_sample, replace=False)
    sil = silhouette_score(coords_fit_rad[idx_s], labels_fit[idx_s])
    db  = davies_bouldin_score(coords_fit_rad[idx_s], labels_fit[idx_s])
    sil_scores.append(sil)
    db_scores.append(db)

k_list = list(K_RANGE)


## Step 5 — Plot Elbow, Silhouette & Davies-Bouldin


In [6]:
fig_eval = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Elbow Method — Inertia (WCSS)</b>',
        '<b>Silhouette Score (higher = better)</b>',
        '<b>Davies-Bouldin Index (lower = better)</b>'
    ]
)

fig_eval.add_trace(go.Scatter(
    x=k_list, y=inertias,
    mode='lines+markers',
    line=dict(color='#065A82', width=2),
    marker=dict(size=8, color='#065A82'),
    name='Inertia',
    hovertemplate='k=%{x}<br>Inertia: %{y:.4f}<extra></extra>'
), row=1, col=1)

best_sil_k = k_list[sil_scores.index(max(sil_scores))]
fig_eval.add_trace(go.Scatter(
    x=k_list, y=sil_scores,
    mode='lines+markers',
    line=dict(color='#02C39A', width=2),
    marker=dict(size=8, color='#02C39A'),
    name='Silhouette',
    hovertemplate='k=%{x}<br>Silhouette: %{y:.4f}<extra></extra>'
), row=1, col=2)
fig_eval.add_vline(x=best_sil_k, line_dash='dash', line_color='#F4A261',
    annotation_text=f'Peak k={best_sil_k}',
    annotation_position='top right', row=1, col=2)

best_db_k = k_list[db_scores.index(min(db_scores))]
fig_eval.add_trace(go.Scatter(
    x=k_list, y=db_scores,
    mode='lines+markers',
    line=dict(color='#E24B4A', width=2),
    marker=dict(size=8, color='#E24B4A'),
    name='Davies-Bouldin',
    hovertemplate='k=%{x}<br>DB Index: %{y:.4f}<extra></extra>'
), row=1, col=3)
fig_eval.add_vline(x=best_db_k, line_dash='dash', line_color='#F4A261',
    annotation_text=f'Trough k={best_db_k}',
    annotation_position='top right', row=1, col=3)

fig_eval.update_layout(
    title=dict(text='<b>Phase 3 — K-Means Evaluation Metrics (k=2 to 12)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=430, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig_eval.update_xaxes(title_text='Number of clusters (k)',
                       showgrid=True, gridcolor='#EEEEEE', dtick=1)
fig_eval.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig_eval.update_yaxes(title_text='Inertia (WCSS)',     row=1, col=1)
fig_eval.update_yaxes(title_text='Silhouette score',   row=1, col=2)
fig_eval.update_yaxes(title_text='Davies-Bouldin idx', row=1, col=3)
fig_eval.write_html('phase5_evaluation_metrics.html')
fig_eval.show()
print(f'Best k by Silhouette   : k={best_sil_k}  (score={max(sil_scores):.4f})')
print(f'Best k by Davies-Bouldin: k={best_db_k}  (score={min(db_scores):.4f})')

Best k by Silhouette   : k=3  (score=0.8276)
Best k by Davies-Bouldin: k=3  (score=0.2516)


## Step 6 — Pick Optimal k


In [7]:
# Cross-check silhouette peak, Davies-Bouldin trough, and the expected 6-zone table
print(f'Silhouette peak: k={best_sil_k}   Davies-Bouldin trough: k={best_db_k}   Expected zones: k=6')

if best_sil_k == best_db_k:
    k_optimal = best_sil_k
elif abs(best_sil_k - 6) <= abs(best_db_k - 6):
    k_optimal = best_sil_k
else:
    k_optimal = 6

print(f'k_optimal = {k_optimal}')


Silhouette peak: k=3   Davies-Bouldin trough: k=3   Expected zones: k=6
k_optimal = 3


## Step 7 — Fit Final K-Means Model


In [8]:
kmeans_final = KMeans(
    n_clusters=k_optimal,
    random_state=SEED,
    n_init=10,
    max_iter=300,
    init='k-means++'
)
kmeans_final.fit(coords_fit_rad)

centres_deg = np.degrees(scaler.inverse_transform(kmeans_final.cluster_centers_))
for i, c in enumerate(centres_deg):
    print(f'Cluster {i}: lat={c[0]:.4f}N  lon={c[1]:.4f}')

labels_fit_final = kmeans_final.labels_
n_sample = min(5000, len(coords_fit_rad))
idx_s = np.random.RandomState(SEED).choice(len(coords_fit_rad), size=n_sample, replace=False)
sil_final = silhouette_score(coords_fit_rad[idx_s], labels_fit_final[idx_s])
db_final  = davies_bouldin_score(coords_fit_rad[idx_s], labels_fit_final[idx_s])
print(f'Silhouette: {sil_final:.4f}   Davies-Bouldin: {db_final:.4f}')


Cluster 0: lat=50.1297N  lon=2.9721
Cluster 1: lat=31.3833N  lon=-9.9230
Cluster 2: lat=16.1587N  lon=-16.6799
Silhouette: 0.8276   Davies-Bouldin: 0.2516


## Step 8 — Haversine Distance Sanity Check


In [9]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = (sin(dlat/2)**2 +
         cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2)
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

sample_idx = np.random.RandomState(SEED).choice(len(coords_fit), size=2000, replace=False)
labels_sample = kmeans_final.predict(coords_fit_rad[sample_idx])

haversine_dists = []
euclidean_dists = []

for idx, lbl in zip(sample_idx, labels_sample):
    lat, lon = coords_fit[idx]
    c_lat, c_lon = centres_deg[lbl]
    hav = haversine_km(lat, lon, c_lat, c_lon)
    point_rad  = scaler.inverse_transform(coords_fit_rad[idx].reshape(1, -1))[0]
    centre_rad = scaler.inverse_transform(kmeans_final.cluster_centers_[lbl].reshape(1, -1))[0]
    euc = np.linalg.norm(point_rad - centre_rad) * 6371
    haversine_dists.append(hav)
    euclidean_dists.append(euc)

hav_arr = np.array(haversine_dists)
euc_arr = np.array(euclidean_dists)
pct_diff = np.abs(hav_arr - euc_arr) / hav_arr * 100

# Radian-based Euclidean K-Means is only valid if this stays under ~5% mean error
print(f'Mean haversine dist: {hav_arr.mean():.2f} km   Mean euclidean-radian dist: {euc_arr.mean():.2f} km')
print(f'Mean % difference: {pct_diff.mean():.2f}%   Max: {pct_diff.max():.2f}%')


Mean haversine dist: 221.48 km   Mean euclidean-radian dist: 231.22 km
Mean % difference: 12.15%   Max: 56.00%


## Step 9 — Assign cluster_id to ALL 61,920 Records


In [10]:
all_coords_rad = scaler.transform(np.radians(df_clean[['latitude','longitude']].values))
df_clean['cluster_id'] = kmeans_final.predict(all_coords_rad)

dist = df_clean['cluster_id'].value_counts().sort_index()
for cid, cnt in dist.items():
    pct  = cnt/len(df_clean)*100
    clat = centres_deg[cid][0]
    print(f'Cluster {cid}: {cnt:6,} records ({pct:5.1f}%)  centroid lat={clat:.2f}N')

min_sz, max_sz = dist.min(), dist.max()
print(f'Cluster size ratio (max/min): {max_sz/min_sz:.2f}x')


Cluster 0: 18,967 records ( 30.6%)  centroid lat=50.13N
Cluster 1: 15,407 records ( 24.9%)  centroid lat=31.38N
Cluster 2: 27,546 records ( 44.5%)  centroid lat=16.16N
Cluster size ratio (max/min): 1.79x


## Step 10 — Per-Point Silhouette: Flag Borderline/Transition Points


In [11]:
n_sil_sample = min(10000, len(all_coords_rad))
sil_idx = np.random.RandomState(SEED).choice(
    len(all_coords_rad), size=n_sil_sample, replace=False
)
sil_sample_labels = df_clean['cluster_id'].values[sil_idx]

per_point_sil = silhouette_samples(
    all_coords_rad[sil_idx], sil_sample_labels
)

borderline_mask = per_point_sil < 0.0
transition_mask = (per_point_sil >= 0.0) & (per_point_sil < 0.2)

print(f'Mean silhouette: {per_point_sil.mean():.4f}')
print(f'Borderline (sil<0.0): {borderline_mask.sum():,} ({borderline_mask.sum()/n_sil_sample*100:.1f}%)')
print(f'Transition (0.0-0.2): {transition_mask.sum():,} ({transition_mask.sum()/n_sil_sample*100:.1f}%)')

fig_sil = go.Figure(go.Histogram(
    x=per_point_sil, nbinsx=50,
    marker_color='#065A82', opacity=0.8,
    hovertemplate='Silhouette: %{x:.3f}<br>Count: %{y}<extra></extra>'
))
fig_sil.add_vline(x=0.0, line_dash='dash', line_color='#E24B4A',
    annotation_text='Borderline boundary (0.0)',
    annotation_position='top right')
fig_sil.add_vline(x=per_point_sil.mean(), line_dash='dot', line_color='#F4A261',
    annotation_text=f'Mean: {per_point_sil.mean():.3f}',
    annotation_position='top left')
fig_sil.update_layout(
    title=dict(text='<b>Per-Point Silhouette Distribution</b>',
               font=dict(size=16, color='#065A82'), x=0.5),
    xaxis_title='Silhouette score',
    yaxis_title='Count',
    height=380, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=80, b=60, l=60, r=60)
)
fig_sil.write_html('phase5_per_point_silhouette.html')
fig_sil.show()


Mean silhouette: 0.8406
Borderline (sil<0.0): 0 (0.0%)
Transition (0.0-0.2): 90 (0.9%)


## Step 11 — Geographic Map: Visual Sanity Check


In [12]:
CLUSTER_COLORS = [
    '#065A82', '#02C39A', '#F4A261', '#E24B4A',
    '#7B68EE', '#2ECC71', '#E67E22', '#9B59B6',
    '#1ABC9C', '#E74C3C', '#3498DB', '#F39C12'
]

fig_map = go.Figure()

for cid in sorted(df_clean['cluster_id'].unique()):
    cdf = df_clean[df_clean['cluster_id'] == cid]
    n_sample = min(3000, len(cdf))
    idx = np.random.RandomState(SEED).choice(len(cdf), size=n_sample, replace=False)
    cdf_s = cdf.iloc[idx]
    color = CLUSTER_COLORS[cid % len(CLUSTER_COLORS)]

    fig_map.add_trace(go.Scattergeo(
        lat=cdf_s['latitude'],
        lon=cdf_s['longitude'],
        mode='markers',
        marker=dict(size=3, color=color, opacity=0.6),
        name=f'Cluster {cid} (n={len(cdf):,})',
        hovertemplate=(
            f'Cluster {cid}<br>'
            'Lat: %{lat:.4f}°<br>Lon: %{lon:.4f}°<extra></extra>'
        )
    ))

fig_map.add_trace(go.Scattergeo(
    lat=centres_deg[:,0],
    lon=centres_deg[:,1],
    mode='markers+text',
    marker=dict(size=18, color='white', symbol='star',
                line=dict(width=2, color='black')),
    text=[f'C{i}' for i in range(k_optimal)],
    textposition='top right',
    textfont=dict(size=12, color='black'),
    name='Centroids',
    hovertemplate='Centroid %{text}<br>Lat: %{lat:.3f}°<br>Lon: %{lon:.3f}°<extra></extra>'
))

fig_map.update_layout(
    title=dict(
        text=f'<b>Phase 5 — K-Means Cluster Map (k={k_optimal})</b><br>'
             '<sup>GPS corridor divided into geographic migration zones · ★ = centroids</sup>',
        font=dict(size=18, color='#065A82'), x=0.5
    ),
    geo=dict(
        showland=True,        landcolor='#F5F5F0',
        showocean=True,       oceancolor='#D6EAF8',
        showcoastlines=True,  coastlinecolor='#AAAAAA',
        showcountries=True,   countrycolor='#CCCCCC',
        projection_type='mercator',
        lonaxis=dict(range=[-22, 12]),
        lataxis=dict(range=[8, 58]),
        bgcolor='#F8FBFD'
    ),
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    height=700, paper_bgcolor='white',
    margin=dict(t=110, b=20, l=20, r=20)
)
fig_map.write_html('phase5_cluster_map.html')
fig_map.show()

## Step 12 — Cluster Profile Table


In [13]:
profiles = []
for cid in sorted(df_clean['cluster_id'].unique()):
    cdf = df_clean[df_clean['cluster_id'] == cid]
    mean_lat  = cdf['latitude'].mean()
    mean_lon  = cdf['longitude'].mean()
    count     = len(cdf)
    rest_pct  = cdf['is_resting'].mean() * 100

    region = 'Unknown'
    for z in EXPECTED_ZONES:
        if z['lat_band'][0] <= mean_lat <= z['lat_band'][1]:
            region = z['name']
            break

    profiles.append({'cluster_id': cid, 'mean_lat': mean_lat,
                     'mean_lon': mean_lon, 'count': count,
                     'rest_pct': rest_pct, 'region': region})

profiles_df = pd.DataFrame(profiles)
display(profiles_df.round(2))

matched = sum(1 for p in profiles if p['region'] != 'Unknown')
print(f'Clusters matching expected zones: {matched}/{k_optimal}')


,cluster_id,mean_lat,mean_lon,count,rest_pct,region
0,0,50.18,3.04,18967,49.19,Netherlands
1,1,31.14,-9.89,15407,45.39,North Africa
2,2,15.98,-16.68,27546,42.47,West Africa


Clusters matching expected zones: 3/3


## Step 14 — Save Output: bird_migration_clustered.csv


In [15]:
df_clean['date_time'] = pd.to_datetime(df_clean['date_time'], utc=True)
output_cols = [
    'date_time', 'bird_name', 'latitude', 'longitude',
    'altitude', 'altitude_clipped', 'speed_2d', 'direction',
    'month', 'hour', 'season', 'is_resting',
    'cluster_id'
]

output_cols = [c for c in output_cols if c in df_clean.columns]
df_output = df_clean[output_cols].copy()

df_output.to_csv('bird_migration_clustered.csv', index=False)

print(f'cluster_id: {sorted(df_output["cluster_id"].unique())} ({k_optimal} zones)')


cluster_id: [np.int32(0), np.int32(1), np.int32(2)] (3 zones)
